# Stage 6. Extract features

## Notebook Information

**Purpose:** Extract per-object morphology, intensity, and texture features from segmented field-of-view images.

**Workflow summary:** Load configuration and metadata, open illumination-corrected FOV images and matching segmentation masks, remove edge-touching labels, compute regionprops/custom intensity features plus Hessian and structure-tensor features, then save one feature table per FOV and update processing metadata.

**Run scope:** Processes all rows in the selected metadata dataframe by default. The expected input is the Part 5 metadata containing illumination-correction and segmentation file references.

**Reproducibility:** Use the configuration/parameter cell as the source of truth for paths and run parameters. For a fresh execution, run sections in order and make sure the metadata input points to the intended Part 5 metadata file rather than a previously generated Part 6 metadata file.

### Authors

| Name | Affiliation |
|---|---|
| Alessandro Ulivi | Infectious Diseases Imaging Platform, Center for Integrative Infectious Disease Research, Heidelberg |
| Edwin Carreno | Scientific Software Center, Heidelberg |
| Christine Schultz | Scientific Software Center, Heidelberg |
| Name Surname | Infectious Diseases Imaging Platform, Center for Integrative Infectious Disease Research, Heidelberg |

## 1. Setup

### 1.1 Imports

In [ ]:
# Built-in imports
import datetime
import os
import logging
from importlib.metadata import version
from pathlib import Path

# Third-party imports
import napari
import numpy as np
import pandas as pd
import tifffile
from omegaconf import OmegaConf
from scipy.ndimage import gaussian_filter
from skimage.transform import resize
from skimage.measure import regionprops_table

# Package imports
from acid.utils.filesystem.filesystem import create_output_directories
from acid.utils.listdirNHF import listdirNHF
from acid.utils.metadata.loading import load_metadata
from acid.utils.get_defaults import default_file_name
from acid.utils.label_image_utils import exclude_label_on_edge
from acid.feature_extraction.default_regionprops import default_regionpros_props
from acid.feature_extraction.extra_regionprops import regionpros_extra_props
from acid.feature_extraction.measure_hessian_matrix import MeasureHessianMatrix
from acid.feature_extraction.measure_structure_tensor import MeasureStructureTensor

### 1.2 Configure logging

In [ ]:
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s",
)

### 1.3 Load configuration

In [ ]:
CONFIG_PATH_FILE = Path("config.yaml")

if not CONFIG_PATH_FILE.is_file():
    CONFIG_PATH_FILE = Path("notebooks/config.yaml")

CONFIG_PATH_FILE = CONFIG_PATH_FILE.resolve()
config = OmegaConf.load(CONFIG_PATH_FILE)
for key, value in config.shared.paths.items():
    config.shared.paths[key] = str((CONFIG_PATH_FILE.parent / value).resolve())

In [ ]:
paths_cfg = config.shared.paths

# Metadata config
metadata_cfg = config.feature_extraction.metadata

## 2. Input Data Preparation

### 2.1 Create output directories

In [ ]:
output_path, _ = create_output_directories(
    output_directory=paths_cfg.feature_tables_dir,
    enable_secondary_output=False,
)

### 2.2 Load metadata dataframe

In [ ]:
metadata_df, metadata_file_name = load_metadata(
    metadata_config=config["feature_extraction"]["metadata"]
)

## 3. Extract features

- Preprocess image and segmentation mask
- Extract features
- Save results

In [ ]:
# FUTURE: src/acid/features_extraction/extract_features_extraction.py

from datetime import datetime
from pathlib import Path

import pandas as pd
import tifffile
from scipy.ndimage import gaussian_filter
from skimage.measure import regionprops_table
from tqdm.notebook import tqdm


def load_tiff(file_path, **kwargs):
    """Load a TIFF image from disk."""
    return tifffile.imread(file_path, **kwargs)


def get_required_filename(metadata_row, column_name):
    """Return a required filename from a metadata row."""
    filename = metadata_row.get(column_name)

    if pd.isna(filename) or str(filename).strip() == "":
        raise ValueError(f"Missing filename in column {column_name!r}")

    return str(filename).strip()


def get_field_of_view_file(metadata_row, config) -> str:
    field_of_view_file = metadata_row.get(
        config.metadata.dataframe_columns.fov_column_name
    )

    if pd.isna(field_of_view_file) or str(field_of_view_file).strip() == "":
        raise ValueError(
            f"Missing FOV filename in column {config.metadata.dataframe_columns.fov_column_name!r}"
        )

    return str(field_of_view_file).strip()


def load_field_of_view(field_of_view_file, corrected_fov_directory):
    """Load one background-corrected field of view."""
    file_path = Path(corrected_fov_directory) / field_of_view_file

    try:
        return load_tiff(file_path)
    except Exception as error:
        raise OSError(f"Could not load field of view TIFF: {file_path}") from error


def load_segmentation_mask(segmentation_file, segmentation_directory):
    """Load one segmentation mask."""
    file_path = Path(segmentation_directory) / segmentation_file

    try:
        return load_tiff(file_path)
    except Exception as error:
        raise OSError(f"Could not load segmentation mask TIFF: {file_path}") from error


def preprocess_field_of_view(field_of_view, config):
    """Prepare image shape for skimage regionprops_table."""
    preprocessed = np.moveaxis(field_of_view, config.processing.channel_axis, -1)

    return gaussian_filter(
        preprocessed,
        sigma=config.processing.sigma,
        axes=config.processing.axes,
    )


def preprocess_segmentation_mask(segmentation):
    """Remove segmented objects touching the image edge."""
    return exclude_label_on_edge(segmentation)


def extract_regionprops_features(
    label_image, intensity_image, properties, extra_properties
):
    """Extract standard and custom regionprops features_extraction."""
    return pd.DataFrame(
        regionprops_table(
            label_image,
            intensity_image=intensity_image,
            properties=properties,
            extra_properties=extra_properties,
        )
    )


def extract_hessian_features(label_image, intensity_image):
    """Extract Hessian matrix eigenvalue features_extraction."""
    hessian_measurer = MeasureHessianMatrix(intensity_image)

    return hessian_measurer.measure_obj_hessian_matrix_eigenval(
        label_image=label_image,
        axis=-1,
    )


def extract_structure_tensor_features(label_image, intensity_image):
    """Extract structure tensor eigenvalue features_extraction."""
    structure_tensor_measurer = MeasureStructureTensor(intensity_image)

    return structure_tensor_measurer.measure_obj_struct_tensor_eigenval(
        label_image=label_image,
        axis=-1,
    )


def merge_feature_tables(
    regionprops_features_df, hessian_features, structure_tensor_features
):
    """Merge all feature tables on object label."""
    merged_features = regionprops_features_df.merge(
        hessian_features,
        on="label",
        how="right",
    )

    return merged_features.merge(
        structure_tensor_features,
        on="label",
        how="right",
    )


def make_features_output_filename(field_of_view_file, config):
    """Create the feature CSV filename for one field of view."""
    stem = str(field_of_view_file).removesuffix(config.features_saving.ome_suffix)

    return f"{stem}{config.features_saving.output_suffix}"


def save_features_dataframe(features_df, output_filename, config, output_directory):
    """Save one feature dataframe as CSV."""
    output_path = Path(output_directory) / output_filename
    output_path.parent.mkdir(parents=True, exist_ok=True)

    features_df.to_csv(
        output_path,
        index=config.features_saving.save_csv_index,
    )

    return output_path

In [ ]:
# FUTURE: src/acid/features_extraction/extract_features_extraction.py


def make_feature_success_result(
    row_index, field_of_view_file, segmentation_file, output_file, config
):
    current_date = datetime.now().strftime(
        config.metadata.dataframe_columns.metadata_df_meta_date_format
    )

    return {
        "row_index": row_index,
        "input_file": field_of_view_file,
        "segmentation_file": segmentation_file,
        "output_file": output_file,
        "success": True,
        "stage": None,
        "error_type": None,
        "error_message": None,
        config.metadata.dataframe_columns.metadata_df_date_clm_name: current_date,
        config.metadata.dataframe_columns.metadata_df_file_name_clm_name: output_file,
        config.metadata.dataframe_columns.metadata_df_method_clm_name: config.metadata.dataframe_columns.preprocessing_steps,
    }


def make_feature_failure_result(
    row_index,
    field_of_view_file,
    segmentation_file,
    error,
    config,
    stage=None,
):
    return {
        "row_index": row_index,
        "input_file": field_of_view_file,
        "segmentation_file": segmentation_file,
        "output_file": None,
        "success": False,
        "stage": stage,
        "error_type": type(error).__name__,
        "error_message": str(error),
        config.metadata.dataframe_columns.metadata_df_date_clm_name: config.metadata.dataframe_columns.null_value,
        config.metadata.dataframe_columns.metadata_df_file_name_clm_name: config.metadata.dataframe_columns.null_value,
        config.metadata.dataframe_columns.metadata_df_method_clm_name: config.metadata.dataframe_columns.null_value,
    }

In [ ]:
# FUTURE: src/acid/features_extraction/extract_features_extraction.py


def extract_features_for_fov(
    row_index,
    metadata_row,
    config,
    properties,
    extra_properties,
    paths,
):
    """Extract and save features for one metadata row."""
    logging.info("---------    ---------")

    # 1. Extract the filename from the row
    field_of_view_file = get_required_filename(
        metadata_row=metadata_row,
        column_name=config.metadata.dataframe_columns.fov_column_name,
    )
    logging.info(f"Field of view filename: {field_of_view_file}")

    # 2. Load the field of view
    field_of_view = load_field_of_view(field_of_view_file, paths.corrected_fov_dir)
    logging.info(f"Working on {field_of_view_file}")

    # 3. Extract the filename from the row
    segmentation_file = get_required_filename(
        metadata_row=metadata_row,
        column_name=config.metadata.dataframe_columns.segmentation_column_name,
    )
    logging.info(f"Mask filename: {segmentation_file}")

    # 4. Load segmentation mask
    segmentation = load_segmentation_mask(
        segmentation_file, paths.segmentation_masks_dir
    )
    logging.info(f"Fetched segmentation file: {segmentation_file}")

    # 5. Preprocess field of view
    preprocessed = preprocess_field_of_view(field_of_view, config)
    logging.info(f"preprocessed fov: {preprocessed.shape}")

    # 6. Preprocess segmentation mask
    label_image = preprocess_segmentation_mask(segmentation)
    logging.info(f"preprocessed label_image: {label_image.shape}")

    # 7. Extract region proposed features
    regionprops_features_df = extract_regionprops_features(
        label_image=label_image,
        intensity_image=preprocessed,
        properties=properties,
        extra_properties=extra_properties,
    )
    logging.info(
        "Region properties features\t | Objects: %d | Features: %d",
        regionprops_features_df.shape[0],
        regionprops_features_df.shape[1],
    )

    # 8. Extract Hessian features
    hessian_features_df = extract_hessian_features(
        label_image=label_image,
        intensity_image=preprocessed,
    )
    logging.info(
        "Hessian features\t\t | Objects: %d | Features: %d",
        hessian_features_df.shape[0],
        hessian_features_df.shape[1],
    )

    # 9. Extract structure tensor features
    structure_tensor_features_df = extract_structure_tensor_features(
        label_image=label_image,
        intensity_image=preprocessed,
    )
    logging.info(
        "Structure tensor features\t | Objects: %d | Features: %d",
        structure_tensor_features_df.shape[0],
        structure_tensor_features_df.shape[1],
    )

    # 10. Merging all features into a single dataframe
    merged_features = merge_feature_tables(
        regionprops_features_df, hessian_features_df, structure_tensor_features_df
    )
    logging.debug("Merged features shape: %s", merged_features.shape)
    logging.info("Feature extraction finished!")

    # 11. Save metadata file with features
    output_filename = make_features_output_filename(field_of_view_file, config)

    output_path = save_features_dataframe(
        features_df=merged_features,
        output_filename=output_filename,
        config=config,
        output_directory=paths.feature_tables_dir,
    )
    logging.info(f"Output file: {output_path}")

    return make_feature_success_result(
        row_index=row_index,
        field_of_view_file=field_of_view_file,
        segmentation_file=segmentation_file,
        output_file=output_filename,
        config=config,
    )


def extract_features_batch(metadata_df, config, paths, max_rows=None):
    row_indices = metadata_df.index

    if max_rows is not None:
        row_indices = metadata_df.index[:max_rows]

    properties = default_regionpros_props()
    extra_properties = regionpros_extra_props()

    results = []

    for row_index in tqdm(row_indices, desc="Extracting features"):
        metadata_row = metadata_df.loc[row_index]
        try:
            result = extract_features_for_fov(
                row_index=row_index,
                metadata_row=metadata_row,
                config=config,
                properties=properties,
                extra_properties=extra_properties,
                paths=paths,
            )
        except Exception as error:
            logging.exception("Feature extraction failed for row %s", row_index)
            result = make_feature_failure_result(
                row_index=row_index,
                field_of_view_file=metadata_row.get(
                    config.metadata.dataframe_columns.fov_column_name
                ),
                segmentation_file=metadata_row.get(
                    config.metadata.dataframe_columns.segmentation_column_name
                ),
                error=error,
                config=config,
                stage="extract_features_for_fov",
            )

        results.append(result)

    return results

In [ ]:
# FUTURE: src/acid/features_extraction/metadata.py

FEATURE_METADATA_COLUMN_CONFIG_KEYS = (
    "metadata_df_date_clm_name",
    "metadata_df_file_name_clm_name",
    "metadata_df_method_clm_name",
)


def get_feature_metadata_columns(config):
    return [getattr(config, key) for key in FEATURE_METADATA_COLUMN_CONFIG_KEYS]


def update_metadata_with_feature_results(
    metadata_df, results, config, copy_dataframe=True
):
    if copy_dataframe:
        metadata_df = metadata_df.copy()

    if not results:
        return metadata_df

    metadata_columns = get_feature_metadata_columns(config)

    results_df = pd.DataFrame.from_records(results).set_index("row_index")

    missing_result_columns = [
        column for column in metadata_columns if column not in results_df.columns
    ]

    if missing_result_columns:
        raise KeyError(
            f"Feature extraction results are missing metadata columns: {missing_result_columns}"
        )

    updates_df = results_df[metadata_columns]

    missing_metadata_columns = [
        column for column in metadata_columns if column not in metadata_df.columns
    ]

    metadata_df = metadata_df.assign(
        **{column: pd.NA for column in missing_metadata_columns}
    )

    metadata_df = metadata_df.astype(dict.fromkeys(metadata_columns, "object"))

    metadata_df.loc[updates_df.index, metadata_columns] = updates_df.to_numpy()

    return metadata_df

In [ ]:
# FUTURE: move function to src/acid/utils/metadata/saving.py
def build_metadata_dataframe_filename(
    metadata_saving_config, project_name, timestamp=None
):
    """Build the standard ACID metadata dataframe filename."""
    if timestamp is None:
        timestamp = datetime.now()

    separator = metadata_saving_config.save_file_name_separator
    metadata_file_suffix = metadata_saving_config.metadata_file_suffix.format(
        save_file_name_separator=separator
    )

    return separator.join(
        [
            timestamp.strftime(metadata_saving_config.metadata_date_format),
            project_name,
            metadata_saving_config.metadata_savingword,
            metadata_file_suffix,
        ]
    )


def build_metadata_dataframe_path(metadata_config, project_name, timestamp=None):
    """Build the full save path for a metadata dataframe."""
    metadata_directory = metadata_config.directory
    metadata_filename = build_metadata_dataframe_filename(
        metadata_saving_config=metadata_config.saving,
        project_name=project_name,
        timestamp=timestamp,
    )

    return Path(metadata_directory) / metadata_filename


def save_metadata_dataframe(metadata_df, metadata_config, project_name, timestamp=None):
    """Save a metadata dataframe as CSV and return the saved path."""

    metadata_path = build_metadata_dataframe_path(
        metadata_config=metadata_config,
        project_name=project_name,
        timestamp=timestamp,
    )

    metadata_path.parent.mkdir(parents=True, exist_ok=True)

    metadata_df.to_csv(
        metadata_path,
        index=metadata_config.saving.save_csv_index,
    )

    return metadata_path

In [ ]:
feature_extraction_cfg = config.feature_extraction

results = extract_features_batch(
    metadata_df=metadata_df, config=feature_extraction_cfg, paths=paths_cfg
)

In [ ]:
results_df = pd.DataFrame.from_records(results)
results_df

## 4. Save results

In [ ]:
metadata_df_updated = update_metadata_with_feature_results(
    metadata_df=metadata_df,
    results=results,
    config=config.feature_extraction.metadata.dataframe_columns,
)

metadata_df_updated.info()

In [ ]:
metadata_path = save_metadata_dataframe(
    metadata_df=metadata_df_updated,
    metadata_config=metadata_cfg,
    project_name=config.project_identity.project_name,
)

logging.info(f"Metadata path: {metadata_path}")
logging.info("processing metadata saved")
logging.info("finished")

# End Notebook